# Build a small RNN to generate smiles codes

Try on two different datasets: a small toy dataset and the smiles column from the ESOL dataset (random sample of 300 points - training time estimated to be ~2-5 min - decrease the sample number if you had issues with runtime before).

In [12]:
import pandas as pd

smiles_list = [
    "CCO",
    "CCN",
    "C=O",
    "C1CCCCC1",
    "CC(=O)O",
    "CCC",
    "COC",
    "CCCl",
    "O"
]

df = pd.read_csv("esol_modified.csv").dropna(subset=["SMILES"]).sample(300)
smiles_list2 = df["SMILES"].to_list()

Build vocabulary and tokeniser

In [13]:
chars = sorted(list(set("".join(smiles_list2))))
stoi = {ch: i for i, ch in enumerate(chars)} #string to index
itos = {i: ch for ch, i in stoi.items()} # index to string

vocab_size = len(chars)

In [14]:
print("stoi: ", stoi)
print("itos:", itos)

stoi:  {' ': 0, '#': 1, '(': 2, ')': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '=': 12, 'B': 13, 'C': 14, 'F': 15, 'H': 16, 'I': 17, 'N': 18, 'O': 19, 'P': 20, 'S': 21, '[': 22, ']': 23, 'c': 24, 'l': 25, 'n': 26, 'o': 27, 'r': 28, 's': 29}
itos: {0: ' ', 1: '#', 2: '(', 3: ')', 4: '1', 5: '2', 6: '3', 7: '4', 8: '5', 9: '6', 10: '7', 11: '8', 12: '=', 13: 'B', 14: 'C', 15: 'F', 16: 'H', 17: 'I', 18: 'N', 19: 'O', 20: 'P', 21: 'S', 22: '[', 23: ']', 24: 'c', 25: 'l', 26: 'n', 27: 'o', 28: 'r', 29: 's'}


In [15]:
def encode(smile):
    return [stoi[c] for c in smile]

data = [encode(s) for s in smiles_list2]

In [16]:
data

[[14, 25, 24, 4, 24, 24, 24, 2, 24, 24, 4, 3, 18, 2, 12, 19, 3, 12, 19],
 [14, 4, 14, 14, 14, 12, 14, 14, 4],
 [19, 14, 14, 2, 19, 3, 14, 2, 19, 3, 14, 19],
 [14, 14, 14, 14, 14, 14, 14, 14, 14, 2, 12, 19, 3, 19, 14],
 [14,
  14,
  18,
  24,
  4,
  26,
  24,
  2,
  14,
  25,
  3,
  26,
  24,
  2,
  18,
  14,
  2,
  14,
  3,
  14,
  3,
  26,
  4],
 [14,
  14,
  14,
  14,
  18,
  2,
  14,
  3,
  14,
  2,
  12,
  19,
  3,
  18,
  24,
  4,
  24,
  24,
  24,
  2,
  14,
  25,
  3,
  24,
  2,
  14,
  25,
  3,
  24,
  4],
 [14, 14, 14, 14, 14, 14, 14, 14, 25],
 [14, 14, 4, 5, 14, 14, 14, 2, 14, 14, 4, 3, 14, 2, 14, 3, 2, 14, 3, 19, 5, 0],
 [14, 14, 14, 2, 14, 14, 3, 14, 19],
 [14, 19, 24, 4, 26, 24, 26, 24, 5, 26, 24, 24, 26, 24, 4, 5, 0],
 [19,
  12,
  14,
  4,
  18,
  14,
  2,
  12,
  19,
  3,
  18,
  14,
  2,
  12,
  19,
  3,
  14,
  4,
  2,
  14,
  2,
  14,
  3,
  14,
  14,
  14,
  3,
  14,
  14,
  12,
  14],
 [14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 2, 12, 19, 3, 19, 14],
 [14, 14

In [17]:
data = [seq for seq in data if len(seq) > 1]

Build Model

In [18]:
import torch
import torch.nn as nn

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x) # two values are returned by RNN: output and hidden state (not used)
        out = self.fc(out)
        return out

Train the RNN

In [19]:
model = SmilesRNN(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    total_loss = 0

    for seq in data:
        x = torch.tensor(seq[:-1], dtype=torch.long).unsqueeze(0)
        y = torch.tensor(seq[1:]).unsqueeze(0)

        output = model(x)
        loss = criterion(output.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.3f}")

Epoch 0, Loss: 497.265
Epoch 20, Loss: 410.654
Epoch 40, Loss: 396.279
Epoch 60, Loss: 390.100
Epoch 80, Loss: 397.108
Epoch 100, Loss: 394.173
Epoch 120, Loss: 402.792
Epoch 140, Loss: 407.761
Epoch 160, Loss: 392.560
Epoch 180, Loss: 409.506


In [20]:
import random

def generate(model, start_char='C', max_len=8):
    model.eval()
    
    idx = torch.tensor([[stoi[start_char]]])
    result = start_char

    for i in range(max_len):
        output = model(idx)
        probs = torch.softmax(output[0, -1], dim=0)
        
        idx_next = torch.multinomial(probs, 1).item()
        char = itos[idx_next]

        result += char
        idx = torch.tensor([[idx_next]])

    return result

for i in range(10):
    print(generate(model))

Cl(CCCl)(
CC=Cc1[nc
CCl)N(CCC
CCCc1[nc1
Cl[nc2c1[
Cl[nc1[nc
CCl)c1[nc
CCCC)(O=O
CCc1[nc3c
CCCN(Fc1[
